In [1]:
import numpy as np
import pandas as pd
import os

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

# Folder containing the datasets saved at the end of Week 3
folder = "Data"

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
data = {}

for i in range(1, 9):

    file_path = os.path.join(
        folder,
        f"function_{i}.csv"
    )

    data[i] = pd.read_csv(file_path)

print("All eight datasets loaded successfully.")

All eight datasets loaded successfully.


In [3]:
for i in range(1, 9):

    n_observations = len(data[i])
    n_dimensions = len(data[i].columns) - 1

    print(
        f"Function {i}: "
        f"{n_observations} observations, "
        f"{n_dimensions} dimensions"
    )

Function 1: 12 observations, 2 dimensions
Function 2: 12 observations, 2 dimensions
Function 3: 17 observations, 3 dimensions
Function 4: 32 observations, 4 dimensions
Function 5: 22 observations, 4 dimensions
Function 6: 22 observations, 5 dimensions
Function 7: 32 observations, 6 dimensions
Function 8: 42 observations, 8 dimensions


In [4]:
# Points submitted last week
new_inputs = {
    1: [0.000045, 0.003062],
    2: [0.709627, 0.881953],
    3: [0.732035, 0.997300, 0.015653],
    4: [0.391976, 0.424757, 0.494823, 0.464424],
    5: [0.671629, 0.981118, 0.990458, 0.985189],
    6: [0.013758, 0.147930, 0.445597, 0.994845, 0.013469],
    7: [0.021549, 0.313334, 0.360216, 0.141732, 0.295328, 0.686548],
    8: [0.025529, 0.451238, 0.098209, 0.031391,
        0.412421, 0.138573, 0.114934, 0.994361]
}

# Outputs returned this week
new_outputs = {
    1: 1.909096326648326e-246,
    2: 0.5486083534111039,
    3: -0.11514149202782288,
    4: -1.850268223115655,
    5: 4595.479318470633,
    6: -1.182842192025624,
    7: 2.1395513389520344,
    8: 9.6453009842564
}

In [5]:
for i in range(1, 9):

    input_columns = [
        col for col in data[i].columns
        if col != "output"
    ]

    new_row = dict(
        zip(input_columns, new_inputs[i])
    )

    new_row["output"] = new_outputs[i]

    data[i] = pd.concat(
        [
            data[i],
            pd.DataFrame([new_row])
        ],
        ignore_index=True
    )

print("All eight new observations added successfully.")

All eight new observations added successfully.


In [6]:
# Compare this week's outputs with the previous best

for i in range(1, 9):

    # Everything except the newly added final row
    previous_data = data[i].iloc[:-1]

    previous_best = previous_data["output"].max()
    new_output = new_outputs[i]

    updated_best = data[i]["output"].max()

    difference = new_output - previous_best

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Previous best: {previous_best:.6f}")
    print(f"New output:    {new_output:.6f}")
    print(f"Difference:    {difference:.6f}")
    print(f"Updated best:  {updated_best:.6f}")

    if new_output > previous_best:
        print("Result: NEW BEST")
    else:
        print("Result: Did not improve")

    print()

FUNCTION 1
Previous best: 0.000000
New output:    0.000000
Difference:    -0.000000
Updated best:  0.000000
Result: Did not improve

FUNCTION 2
Previous best: 0.611205
New output:    0.548608
Difference:    -0.062597
Updated best:  0.611205
Result: Did not improve

FUNCTION 3
Previous best: -0.020167
New output:    -0.115141
Difference:    -0.094975
Updated best:  -0.020167
Result: Did not improve

FUNCTION 4
Previous best: -0.300645
New output:    -1.850268
Difference:    -1.549624
Updated best:  -0.300645
Result: Did not improve

FUNCTION 5
Previous best: 2765.819787
New output:    4595.479318
Difference:    1829.659532
Updated best:  4595.479318
Result: NEW BEST

FUNCTION 6
Previous best: -0.458511
New output:    -1.182842
Difference:    -0.724332
Updated best:  -0.458511
Result: Did not improve

FUNCTION 7
Previous best: 1.950612
New output:    2.139551
Difference:    0.188939
Updated best:  2.139551
Result: NEW BEST

FUNCTION 8
Previous best: 9.821986
New output:    9.645301
Diffe

In [7]:
# Refit Gaussian Process models using all available data

models_week4 = {}

for i in range(1, 9):

    X = data[i].drop(columns="output").values
    y = data[i]["output"].values

    kernel = RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-5, 1e5)
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y)

    models_week4[i] = gp

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)
    print(f"Fitted kernel: {gp.kernel_}")
    print()

FUNCTION 1
Fitted kernel: RBF(length_scale=1e-05)

FUNCTION 2
Fitted kernel: RBF(length_scale=0.0516)

FUNCTION 3
Fitted kernel: RBF(length_scale=0.198)

FUNCTION 4
Fitted kernel: RBF(length_scale=0.428)

FUNCTION 5
Fitted kernel: RBF(length_scale=0.367)

FUNCTION 6
Fitted kernel: RBF(length_scale=1e-05)

FUNCTION 7
Fitted kernel: RBF(length_scale=0.296)

FUNCTION 8
Fitted kernel: RBF(length_scale=0.949)



In [8]:
# Diagnose GP behaviour for Functions 1 and 6

rng = np.random.default_rng(42)

for i in [1, 6]:

    X = data[i].drop(columns="output").values
    n_dimensions = X.shape[1]

    X_candidates = rng.uniform(
        0,
        1,
        size=(100000, n_dimensions)
    )

    mean, std = models_week4[i].predict(
        X_candidates,
        return_std=True
    )

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Predicted mean minimum: {mean.min():.10f}")
    print(f"Predicted mean maximum: {mean.max():.10f}")
    print(f"Predicted mean range:   {mean.max() - mean.min():.10f}")

    print()

    print(f"Uncertainty minimum:    {std.min():.10f}")
    print(f"Uncertainty maximum:    {std.max():.10f}")
    print(f"Uncertainty range:      {std.max() - std.min():.10f}")

    print()

FUNCTION 1
Predicted mean minimum: -0.0002773894
Predicted mean maximum: -0.0000000154
Predicted mean range:   0.0002773740

Uncertainty minimum:    0.0000101201
Uncertainty maximum:    0.0009609052
Uncertainty range:      0.0009507851

FUNCTION 6
Predicted mean minimum: -1.4874220585
Predicted mean maximum: -1.4874220585
Predicted mean range:   0.0000000000

Uncertainty minimum:    0.5341917888
Uncertainty maximum:    0.5341917888
Uncertainty range:      0.0000000000



In [9]:
# Compare kappa = 1.5, 2.5 and 3.5

kappa_values = [1.5, 2.5, 3.5]
kappa_functions = [2, 3, 4, 5, 7, 8]

n_candidates = 100000

kappa_results = {}

for i in kappa_functions:

    X_existing = data[i].drop(columns="output").values
    y_existing = data[i]["output"].values

    gp = models_week4[i]
    n_dimensions = X_existing.shape[1]

    # Use the same candidate set for all kappa values
    rng = np.random.default_rng(42 + i)

    X_candidates = rng.uniform(
        0,
        1,
        size=(n_candidates, n_dimensions)
    )

    # GP predictions only need to be calculated once
    mean, std = gp.predict(
        X_candidates,
        return_std=True
    )

    # Current best observation
    best_index = np.argmax(y_existing)
    current_best_point = X_existing[best_index]
    current_best_output = y_existing[best_index]

    kappa_results[i] = {}

    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    print(f"Current best output: {current_best_output:.6f}")
    print()

    for kappa in kappa_values:

        ucb = mean + kappa * std
        selected_index = np.argmax(ucb)

        point = X_candidates[selected_index]

        distance_best = np.linalg.norm(
            point - current_best_point
        )

        distances_existing = np.linalg.norm(
            X_existing - point,
            axis=1
        )

        nearest_distance = distances_existing.min()

        kappa_results[i][kappa] = {
            "point": point,
            "predicted_mean": mean[selected_index],
            "uncertainty": std[selected_index],
            "ucb": ucb[selected_index],
            "distance_best": distance_best,
            "nearest_distance": nearest_distance
        }

        print(f"KAPPA = {kappa}")
        print("Selected point:")
        print(np.round(point, 6))

        print(f"Predicted output:               {mean[selected_index]:.6f}")
        print(f"Uncertainty:                    {std[selected_index]:.6f}")
        print(f"UCB:                            {ucb[selected_index]:.6f}")
        print(f"Distance from current best:     {distance_best:.4f}")
        print(f"Distance from nearest existing: {nearest_distance:.4f}")

        print()

FUNCTION 2
Current best output: 0.611205

KAPPA = 1.5
Selected point:
[0.73856  0.912272]
Predicted output:               0.556031
Uncertainty:                    0.121843
UCB:                            0.738795
Distance from current best:     0.0387
Distance from nearest existing: 0.0387

KAPPA = 2.5
Selected point:
[0.688986 0.063026]
Predicted output:               0.375397
Uncertainty:                    0.210671
UCB:                            0.902075
Distance from current best:     0.8636
Distance from nearest existing: 0.0652

KAPPA = 3.5
Selected point:
[0.725891 0.170323]
Predicted output:               0.342377
Uncertainty:                    0.221941
UCB:                            1.119170
Distance from current best:     0.7566
Distance from nearest existing: 0.0759

FUNCTION 3
Current best output: -0.020167

KAPPA = 1.5
Selected point:
[0.41795  0.732195 0.005457]
Predicted output:               -0.033932
Uncertainty:                    0.053728
UCB:                     

In [10]:
# Compare distances between candidates selected by each kappa value

for i in kappa_functions:

    p15 = kappa_results[i][1.5]["point"]
    p25 = kappa_results[i][2.5]["point"]
    p35 = kappa_results[i][3.5]["point"]

    d_15_25 = np.linalg.norm(p15 - p25)
    d_15_35 = np.linalg.norm(p15 - p35)
    d_25_35 = np.linalg.norm(p25 - p35)

    print("=" * 60)
    print(f"FUNCTION {i}")
    print("=" * 60)

    print(f"Distance κ 1.5 → 2.5: {d_15_25:.4f}")
    print(f"Distance κ 1.5 → 3.5: {d_15_35:.4f}")
    print(f"Distance κ 2.5 → 3.5: {d_25_35:.4f}")

    print()

FUNCTION 2
Distance κ 1.5 → 2.5: 0.8507
Distance κ 1.5 → 3.5: 0.7421
Distance κ 2.5 → 3.5: 0.1135

FUNCTION 3
Distance κ 1.5 → 2.5: 0.6195
Distance κ 1.5 → 3.5: 0.6647
Distance κ 2.5 → 3.5: 0.0707

FUNCTION 4
Distance κ 1.5 → 2.5: 0.0478
Distance κ 1.5 → 3.5: 0.9597
Distance κ 2.5 → 3.5: 0.9480

FUNCTION 5
Distance κ 1.5 → 2.5: 0.1438
Distance κ 1.5 → 3.5: 0.1438
Distance κ 2.5 → 3.5: 0.0000

FUNCTION 7
Distance κ 1.5 → 2.5: 0.0000
Distance κ 1.5 → 3.5: 0.1986
Distance κ 2.5 → 3.5: 0.1986

FUNCTION 8
Distance κ 1.5 → 2.5: 0.7627
Distance κ 1.5 → 3.5: 0.9229
Distance κ 2.5 → 3.5: 0.9939



In [11]:
from scipy.stats import norm

# Expected Improvement acquisition function
def expected_improvement(mean, std, best_y, xi=0.01):

    improvement = mean - best_y - xi

    with np.errstate(divide="ignore", invalid="ignore"):
        Z = improvement / std

        ei = (
            improvement * norm.cdf(Z)
            + std * norm.pdf(Z)
        )

    # EI should be zero where uncertainty is zero
    ei[std == 0.0] = 0.0

    return ei

In [12]:
# Compare Expected Improvement with provisional UCB selections

ei_results = {}

for i in kappa_functions:

    X_existing = data[i].drop(columns="output").values
    y_existing = data[i]["output"].values

    gp = models_week4[i]
    n_dimensions = X_existing.shape[1]

    # Recreate the same candidate set used in Step 3
    rng = np.random.default_rng(42 + i)

    X_candidates = rng.uniform(
        0,
        1,
        size=(100000, n_dimensions)
    )

    mean, std = gp.predict(
        X_candidates,
        return_std=True
    )

    # Current best
    best_index = np.argmax(y_existing)
    current_best_point = X_existing[best_index]
    current_best_output = y_existing[best_index]

    # Calculate Expected Improvement
    ei = expected_improvement(
        mean,
        std,
        current_best_output,
        xi=0.01
    )

    selected_index = np.argmax(ei)
    point = X_candidates[selected_index]

    # Distances
    distance_best = np.linalg.norm(
        point - current_best_point
    )

    nearest_distance = np.linalg.norm(
        X_existing - point,
        axis=1
    ).min()

    # Compare with provisional kappa = 1.5 point
    ucb_point = kappa_results[i][1.5]["point"]

    distance_from_ucb = np.linalg.norm(
        point - ucb_point
    )

    ei_results[i] = {
        "point": point,
        "predicted_mean": mean[selected_index],
        "uncertainty": std[selected_index],
        "ei": ei[selected_index],
        "distance_best": distance_best,
        "nearest_distance": nearest_distance,
        "distance_from_ucb": distance_from_ucb
    }

    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    print("EI selected point:")
    print(np.round(point, 6))

    print(f"Predicted output:               {mean[selected_index]:.6f}")
    print(f"Uncertainty:                    {std[selected_index]:.6f}")
    print(f"Expected Improvement:           {ei[selected_index]:.6f}")
    print(f"Distance from current best:     {distance_best:.4f}")
    print(f"Distance from nearest existing: {nearest_distance:.4f}")
    print(f"Distance from κ=1.5 UCB point:  {distance_from_ucb:.4f}")

    print()

FUNCTION 2
EI selected point:
[0.730707 0.915532]
Predicted output:               0.590106
Uncertainty:                    0.095042
Expected Improvement:           0.024378
Distance from current best:     0.0302
Distance from nearest existing: 0.0302
Distance from κ=1.5 UCB point:  0.0085

FUNCTION 3
EI selected point:
[0.433866 0.716516 0.000863]
Predicted output:               -0.031179
Uncertainty:                    0.051758
Expected Improvement:           0.011821
Distance from current best:     0.6086
Distance from nearest existing: 0.1788
Distance from κ=1.5 UCB point:  0.0228

FUNCTION 4
EI selected point:
[0.44374  0.493644 0.40979  0.452219]
Predicted output:               -0.309086
Uncertainty:                    0.456642
Expected Improvement:           0.173102
Distance from current best:     0.0918
Distance from nearest existing: 0.0918
Distance from κ=1.5 UCB point:  0.0000

FUNCTION 5
EI selected point:
[0.851064 0.985689 0.919598 0.97933 ]
Predicted output:             

In [13]:
from sklearn.metrics import pairwise_distances

alternative_selected = {}

for i in [1, 6]:

    X_existing = data[i].drop(columns="output").values
    n_dimensions = X_existing.shape[1]

    rng = np.random.default_rng(42 + i)

    X_candidates = rng.uniform(
        0,
        1,
        size=(100000, n_dimensions)
    )

    # Distance from each candidate to all existing observations
    distances = pairwise_distances(
        X_candidates,
        X_existing
    )

    # Distance to nearest existing observation
    nearest_distances = distances.min(axis=1)

    # Select candidate furthest from its nearest observation
    best_index = np.argmax(nearest_distances)

    selected_point = X_candidates[best_index]
    max_distance = nearest_distances[best_index]

    alternative_selected[i] = selected_point

    # Current best point
    best_obs_index = data[i]["output"].idxmax()

    current_best_point = (
        data[i]
        .drop(columns="output")
        .loc[best_obs_index]
        .values
        .astype(float)
    )

    distance_best = np.linalg.norm(
        selected_point - current_best_point
    )

    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    print("Maximum-distance exploration point:")
    print(np.round(selected_point, 6))

    print(
        f"Distance from nearest existing: "
        f"{max_distance:.4f}"
    )

    print(
        f"Distance from current best:      "
        f"{distance_best:.4f}"
    )

    print()

FUNCTION 1
Maximum-distance exploration point:
[0.41743  0.463246]
Distance from nearest existing: 0.3153
Distance from current best:      0.4137

FUNCTION 6
Maximum-distance exploration point:
[0.033216 0.058439 0.124012 0.011504 0.042834]
Distance from nearest existing: 0.8782
Distance from current best:      0.8810



In [14]:
# Final Week 4 query selections

final_points = {
    1: alternative_selected[1],
    2: ei_results[2]["point"],
    3: ei_results[3]["point"],
    4: ei_results[4]["point"],
    5: ei_results[5]["point"],
    6: alternative_selected[6],
    7: ei_results[7]["point"],
    8: ei_results[8]["point"]
}

selection_methods = {
    1: "Maximum-distance exploration",
    2: "Expected Improvement",
    3: "Expected Improvement",
    4: "EI / UCB agreement",
    5: "EI / UCB agreement",
    6: "Maximum-distance exploration",
    7: "Expected Improvement",
    8: "Expected Improvement"
}

In [15]:
# Final validation checks

for i in range(1, 9):

    point = np.asarray(final_points[i])
    X_existing = data[i].drop(columns="output").values

    # Check permitted range
    in_bounds = np.all(
        (point >= 0) & (point <= 1)
    )

    # Distance from all existing observations
    distances = np.linalg.norm(
        X_existing - point,
        axis=1
    )

    nearest_distance = distances.min()

    # Check for duplicate
    duplicate = np.any(
        np.all(
            np.isclose(
                X_existing,
                point,
                atol=1e-8
            ),
            axis=1
        )
    )

    # Current best point
    best_index = data[i]["output"].idxmax()

    best_point = (
        data[i]
        .drop(columns="output")
        .loc[best_index]
        .values
        .astype(float)
    )

    distance_best = np.linalg.norm(
        point - best_point
    )

    print("=" * 70)
    print(f"FUNCTION {i}")
    print("=" * 70)

    print("Selected point:")
    print(np.round(point, 6))

    print(f"Selection method: {selection_methods[i]}")
    print(f"In bounds: {in_bounds}")
    print(f"Duplicate existing point: {duplicate}")
    print(f"Distance to nearest observation: {nearest_distance:.6f}")
    print(f"Distance from current best:      {distance_best:.6f}")

    print()

FUNCTION 1
Selected point:
[0.41743  0.463246]
Selection method: Maximum-distance exploration
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.315336
Distance from current best:      0.413652

FUNCTION 2
Selected point:
[0.730707 0.915532]
Selection method: Expected Improvement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.030161
Distance from current best:      0.030161

FUNCTION 3
Selected point:
[0.433866 0.716516 0.000863]
Selection method: Expected Improvement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.178778
Distance from current best:      0.608589

FUNCTION 4
Selected point:
[0.44374  0.493644 0.40979  0.452219]
Selection method: EI / UCB agreement
In bounds: True
Duplicate existing point: False
Distance to nearest observation: 0.091845
Distance from current best:      0.091845

FUNCTION 5
Selected point:
[0.851064 0.985689 0.919598 0.97933 ]
Selection method: EI / UCB

In [16]:
for i in range(1, 9):
    submission = "-".join(
        f"{value:.6f}"
        for value in final_points[i]
    )
    print(f"Function {i}: {submission}")

Function 1: 0.417430-0.463246
Function 2: 0.730707-0.915532
Function 3: 0.433866-0.716516-0.000863
Function 4: 0.443740-0.493644-0.409790-0.452219
Function 5: 0.851064-0.985689-0.919598-0.979330
Function 6: 0.033216-0.058439-0.124012-0.011504-0.042834
Function 7: 0.154752-0.313081-0.341855-0.169387-0.318877-0.633022
Function 8: 0.148540-0.297539-0.201860-0.328776-0.636564-0.500531-0.197949-0.479426
